In [ ]:
"""
    A practical example for fine-tuning an open-weight LLM using LoRA with Hugging Face Transformers and PEFT (Parameter-Efficient Fine-Tuning)

    We will use:
        - LLaMA-style workflow
        - LoRA (efficient fine-tuning)
        - Small instruction dataset
        - Single GPU setup
"""

In [ ]:
import os
import torch
import warnings

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Torch Version:", torch.__version__)
print("CUDA Availablility:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

import torch

def print_gpu_memory(tag="GPU Status"):
    if not torch.cuda.is_available():
        print("CUDA not available")
        return

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    free, total = torch.cuda.mem_get_info()
    free = free / 1024**3
    total = total / 1024**3
    used = total - free

    print(f"\n[{tag}]")
    print(f"Allocated: {allocated:.3f} GB")
    print(f"Reserved : {reserved:.3f} GB")
    print(f"Free     : {free:.3f} GB")
    print(f"Used     : {used:.3f} GB / {total:.3f} GB")

print_gpu_memory("Initial Status")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model
)

model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="cuda:0",
    dtype=torch.bfloat16
)

model = prepare_model_for_kbit_training(model)
print("Model is loaded on:", model.device)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

model.print_trainable_parameters()

In [ ]:
import json
import random
from datasets import Dataset

repeat_factor = 2
seed = 42

with open("train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

original_size = len(data)

# Repeat
data = data * repeat_factor

# Shuffle
random.seed(seed)
random.shuffle(data)

dataset = Dataset.from_list(data)

print("Original number of samples:", original_size)
print("Repeat factor:", repeat_factor)
print("Effective number of samples:", len(dataset))

In [ ]:
def formatting_prompts_func(examples):
    texts = []

    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )

        texts.append(text)

    return {"text": texts}


dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset.column_names
)

In [ ]:
import json
import torch
from transformers import TrainerCallback

class TrainingLogger(TrainerCallback):
    def __init__(self, save_path="train_logs.jsonl"):
        self.save_path = save_path

        # reset file
        with open(self.save_path, "w", encoding="utf-8") as f:
            pass

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        # only log meaningful training step
        if "loss" not in logs:
            return

        step = state.global_step

        log_entry = {
            "step": step,
            "loss": logs.get("loss"),
            "learning_rate": logs.get("learning_rate"),
            "grad_norm": logs.get("grad_norm"),
        }

        if torch.cuda.is_available():
            log_entry["gpu_alloc"] = torch.cuda.memory_allocated() / 1024**3
            log_entry["gpu_reserved"] = torch.cuda.memory_reserved() / 1024**3

        with open(self.save_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(log_entry) + "\n")
            f.flush()

logger = TrainingLogger()

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="llm-ft-lora",
    num_train_epochs=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=10,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.0,
    logging_steps=5,
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_length=1024,
    packing=False,
    assistant_only_loss=False,
    report_to="none",
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    callbacks=[logger]
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained("final-lora-adapter")
tokenizer.save_pretrained("final-lora-adapter")
print("Final LoRA adapter is saved!")

In [ ]:
import matplotlib.pyplot as plt
import json

def load_logs(path="train_logs.jsonl"):
    logs = []
    with open(path, "r") as f:
        for line in f:
            logs.append(json.loads(line))
            
    return logs

def prepare(logs):
    steps = [x["step"] for x in logs]
    loss = [x["loss"] for x in logs]
    lr = [x["learning_rate"] for x in logs]
    grad = [x["grad_norm"] for x in logs]
    gpu_a = [x.get("gpu_alloc") for x in logs]
    gpu_r = [x.get("gpu_reserved") for x in logs]

    return steps, loss, lr, grad, gpu_a, gpu_r

def plot_training_from_file(path="train_logs.jsonl"):
    logs = load_logs(path)
    steps, loss, lr, grad, gpu_a, gpu_r = prepare(logs)

    fig, axes = plt.subplots(5, 1, figsize=(8, 12), sharex=True)

    axes[0].plot(steps, loss, color="red")
    axes[0].set_title("Training Loss")

    axes[1].plot(steps, lr, color="blue")
    axes[1].set_title("Learning Rate")

    axes[2].plot(steps, grad, color="green")
    axes[2].set_title("Gradient Norm")

    axes[3].plot(steps, gpu_a, color="purple")
    axes[3].set_title("GPU Allocated Memory")

    axes[4].plot(steps, gpu_r, color="orange")
    axes[4].set_title("GPU Reserved Memory")

    for ax in axes:
        ax.set_facecolor("#f5f5f5") 
        ax.grid(True, which="both", linestyle="--", linewidth=0.5, color="darkgray", alpha=0.7)

    fig.suptitle("Training Summary (LoRA + TRL)", fontsize=14)
    plt.tight_layout()
    plt.savefig("training_summary.png", dpi=300)
    print("Plot successfully saved as 'training_summary.png'")

In [ ]:
plot_training_from_file()

In [ ]:
# Restart Kernel to release CUDA Memory
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

import torch

def print_gpu_memory(tag="GPU Status"):
    if not torch.cuda.is_available():
        print("CUDA not available")
        return

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    free, total = torch.cuda.mem_get_info()
    free = free / 1024**3
    total = total / 1024**3
    used = total - free

    print(f"\n[{tag}]")
    print(f"Allocated: {allocated:.3f} GB")
    print(f"Reserved : {reserved:.3f} GB")
    print(f"Free     : {free:.3f} GB")
    print(f"Used     : {used:.3f} GB / {total:.3f} GB")

print_gpu_memory("After Cleanup")

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer

model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="cuda:0",
    dtype=torch.bfloat16
)

model = PeftModel.from_pretrained(
    base_model,
    "final-lora-adapter"
)

model.eval()

In [ ]:
question = "What information do we get from Z-map?"

messages = [
    {
        "role": "system",
        "content": "You are a technical assistant that answers questions about S******* sensors clearly and accurately."
    },
    {
        "role": "user",
        "content": question
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

input_length = inputs.shape[1] if hasattr(inputs, "shape") else len(inputs[0])
inputs = inputs.to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        temperature=0.0,
        top_p=1.0
    )

generated_answer = tokenizer.decode(
    outputs[0][input_length:], 
    skip_special_tokens=True
)

print("QUESTION:", question)
print("ANSWER:", generated_answer)